# 03 — Calibration (thresholds, temperature, conformal) & weight export

Uses the **true out-of-fold** predictions from notebook 02 (`stage2_oof.npz`):

| Quantity | Fitted on |
|---|---|
| QWK decision thresholds | all OOF scans |
| Temperature *T* | OOF half A |
| Conformal `q_hat` (95 %) | OOF half B, on temperature-scaled probabilities |
| Reported QWK (95 % CI), ECE, coverage, abstention | locked **test** split (4-fold ensemble) |

The pipeline applies exactly these values at inference. There is intentionally **no synthetic fallback**: calibration numbers must come from real predictions.

In [ ]:
# 1. Mount Google Drive, clone/update the repo and unpack the preprocessed data to local disk
import os, sys, subprocess, tarfile, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/Retinopathy')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/dprincepiriyan/Retinopathy.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.8'], check=True)
    LOCAL_DATA_ROOT = Path('/content/data/processed')
    NEEDS_DATA = False
    if NEEDS_DATA and not (LOCAL_DATA_ROOT / 'stage2_grading_splits.csv').exists():
        drive_root = Path('/content/drive/MyDrive')
        candidates = list(drive_root.rglob('retinopathy_processed.tar.gz')) + list(drive_root.rglob('retinopathy_processed.zip'))
        if not candidates:
            raise FileNotFoundError('Upload retinopathy_processed.tar.gz to your Google Drive first.')
        archive_path = candidates[0]
        print(f'Extracting {archive_path} -> /content/data/ ...')
        Path('/content/data').mkdir(parents=True, exist_ok=True)
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path) as zf:
                zf.extractall('/content/data')
        else:
            with tarfile.open(archive_path, 'r:gz') as tf:
                tf.extractall('/content/data', filter='data')
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/Retinopathy_Checkpoints')
else:
    REPO_DIR = Path.cwd().resolve()
    if REPO_DIR.name == 'colab':
        REPO_DIR = REPO_DIR.parent
    LOCAL_DATA_ROOT = REPO_DIR / 'data' / 'processed'
    DRIVE_CKPT_DIR = None

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
LOCAL_CKPT_DIR = REPO_DIR / 'checkpoints'
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_CKPT_DIR is not None:
    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

from src.utils.tpu_accelerator import ColabTPUAccelerator
accel = ColabTPUAccelerator(prefer_tpu=False)  # use a GPU runtime (T4 / L4 / A100)
print(f'Accelerator: {accel.info.device_name} | AMP dtype: {accel.info.amp_dtype} | GradScaler: {accel.grad_scaler is not None}')
if accel.info.backend != 'cuda':
    print('WARNING: no GPU detected - Runtime > Change runtime type > GPU. CPU training will be extremely slow.')

In [ ]:
# 2. Fit calibration from OOF predictions and evaluate it on the locked test split
import json
from src.training.calibration import calibrate_from_files

SRC_DIR = DRIVE_CKPT_DIR or LOCAL_CKPT_DIR
oof_path = SRC_DIR / 'stage2_oof.npz'
if not oof_path.exists():
    raise FileNotFoundError(f'{oof_path} not found - finish all folds in notebook 02 first.')

payload = calibrate_from_files(oof_path, SRC_DIR / 'calibration_params.json', SRC_DIR / 'stage2_test_ensemble.npz', alpha=0.05)
print(json.dumps(payload, indent=2))

In [ ]:
# 3. Download everything the local dashboard needs into checkpoints/
files_needed = ['calibration_params.json', 'stage1_multi_label_best.pth'] + sorted(p.name for p in SRC_DIR.glob('stage2_grader_fold*.pth'))
print('Copy these into <repo>/checkpoints/ on your laptop:', files_needed)
if IN_COLAB:
    from google.colab import files
    for name in files_needed:
        if (SRC_DIR / name).exists():
            files.download(str(SRC_DIR / name))
        else:
            print(f'Missing: {name}')